In [1]:
import os
import sys
import torch
from tqdm import tqdm
from dotenv import load_dotenv
from tqdm import tqdm
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

load_dotenv()
sys.path.append(os.getenv("ROOT_PATH"))
import miso_utils.datasets as mud

def show_tensor_image(tensor):
 
    image = tensor.cpu().clone().detach()
    image = image.permute(1, 2, 0)
    
    if image.min() < 0 or image.max() > 1:
        image = (image - image.min()) / (image.max() - image.min())
    
    plt.imshow(image)
    plt.axis('off')
    plt.show()

In [2]:
import torch
from PIL import Image
import open_clip

clip_model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
clip_tokenizer = open_clip.get_tokenizer('ViT-B-32')

val_data = mud.MisoValDataset(os.getenv("VAL_PATH"), image_transform=preprocess )
train_data = mud.MisoTrainDataset(os.getenv("TRAIN_PATH"), image_transform=preprocess)

Files Left: 600it [00:08, 67.90it/s]
Files Left: 9395it [02:18, 67.93it/s]


In [3]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
val_loader = DataLoader(val_data, batch_size=64, shuffle=False)

In [4]:
class ANNModel(nn.Module):
    def __init__(self, input_dim=1024):
        super(ANNModel, self).__init__()
        
        # A simple architecture: Input -> Hidden -> Output
        self.network = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.2),      # Helps prevent overfitting on smaller datasets
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 1)     # Single output for binary classification
            # Note: No Sigmoid here if using nn.BCEWithLogitsLoss (recommended)
        )

    def forward(self, x):
        return self.network(x)
    
def ClipConcated(images,texts):
    img_emb = clip_model.encode_image(images)
    txt_emb = clip_model.encode_text(texts)
    # Normalize
    img_emb /= img_emb.norm(dim=-1, keepdim=True)
    txt_emb /= txt_emb.norm(dim=-1, keepdim=True)
    return torch.cat((img_emb, txt_emb), dim=1)


In [8]:
def train_miso_model(ann_model, clip_model, clip_tokenizer, train_loader, val_loader, save_path, epochs=10, lr=1e-4, device="cuda"):
    ann_model.to(device)
    clip_model.to(device)
    clip_model.eval() 
    
    # 1. Change reduction to 'sum'
    criterion = nn.BCEWithLogitsLoss(reduction='sum')
    optimizer = torch.optim.Adam(ann_model.parameters(), lr=lr)
    
    best_val_acc = 0.0
    print(f"Starting training on {device}...")

    for epoch in range(epochs):
        # --- TRAINING PHASE ---
        ann_model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
        for batch in pbar:
            images = batch['img'].to(device)
            text_tokens = clip_tokenizer(batch['transcription']).to(device)
            labels = batch['indian_label'].to(device).float()

            optimizer.zero_grad()
            with torch.no_grad(), torch.amp.autocast('cuda'):
                features = ClipConcated(images, text_tokens) 
            
            logits = ann_model(features.float()).squeeze()
            loss = criterion(logits, labels) # This is now the sum of losses in the batch
            loss.backward()
            optimizer.step()

            # 2. Track metrics
            train_loss += loss.item()
            train_total += labels.size(0)
            
            preds = (torch.sigmoid(logits) > 0.5).float()
            train_correct += (preds == labels).sum().item()

            # 3. Display Running Average Loss (Total Loss / Total Samples)
            pbar.set_postfix(
                avg_loss=f"{train_loss/train_total:.4f}", 
                acc=f"{100*train_correct/train_total:.2f}%"
            )

        # --- VALIDATION PHASE ---
        ann_model.eval()
        val_loss, val_total, val_correct = 0.0, 0, 0
        
        with torch.no_grad():
            vbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]")
            for batch in vbar:
                images = batch['img'].to(device)
                text_tokens = clip_tokenizer(batch['transcription']).to(device)
                labels = batch['indian_label'].to(device).float()

                with torch.amp.autocast('cuda'):
                    features = ClipConcated(images, text_tokens)
                
                logits = ann_model(features.float()).squeeze()
                loss = criterion(logits, labels)
                
                val_loss += loss.item()
                val_total += labels.size(0)
                
                preds = (torch.sigmoid(logits) > 0.5).float()
                val_correct += (preds == labels).sum().item()
                
                vbar.set_postfix(
                    avg_loss=f"{val_loss/val_total:.4f}", 
                    acc=f"{100*val_correct/val_total:.2f}%"
                )

        epoch_val_acc = 100 * val_correct / val_total
        if epoch_val_acc > best_val_acc:
            best_val_acc = epoch_val_acc
            torch.save(ann_model.state_dict(), save_path + 'best_miso_model.pt')
            print(f"--> Saved Best Model (Val Acc: {best_val_acc:.2f}%)")

        print("")

    return ann_model

In [9]:
ann_model = ANNModel()
train_miso_model(ann_model,clip_model,clip_tokenizer,train_loader,val_loader,save_path=os.getenv("WEIGHTS_PATH") + "/ClipConcatAnn/",epochs = 50,lr = 1e-4 * 5)

Starting training on cuda...


Epoch 1/50 [Val]: 100%|██████████| 10/10 [00:00<00:00, 14.50it/s, acc=78.17%, avg_loss=0.4499]


--> Saved Best Model (Val Acc: 78.17%)



Epoch 2/50 [Val]: 100%|██████████| 10/10 [00:00<00:00, 13.53it/s, acc=77.83%, avg_loss=0.4458]


Epoch 3/50 [Val]: 100%|██████████| 10/10 [00:00<00:00, 14.00it/s, acc=79.50%, avg_loss=0.4450]


...

Epoch 47/50 [Val]: 100%|██████████| 10/10 [00:00<00:00, 13.24it/s, acc=78.50%, avg_loss=1.4073]


Epoch 48/50 [Val]: 100%|██████████| 10/10 [00:00<00:00, 14.18it/s, acc=78.67%, avg_loss=1.4914]


Epoch 49/50 [Val]: 100%|██████████| 10/10 [00:00<00:00, 13.91it/s, acc=78.17%, avg_loss=1.5400]


Epoch 50/50 [Val]: 100%|██████████| 10/10 [00:00<00:00, 12.74it/s, acc=78.83%, avg_loss=1.5264]

ANNModel(
  (network): Sequential(
    (0): Linear(in_features=1024, out_features=512, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=512, out_features=128, bias=True)
    (4): ReLU()
    (5): Linear(in_features=128, out_features=1, bias=True)
  )
)